# 1. Data Exploration

**Purpose**: Understand data before training. Run once, refer back as needed.

## Goals
1. Load samples from all three tables
2. Field coverage comparison
3. ROR ID overlap analysis (ground truth availability)
4. Blocking rule coverage simulation
5. Name quality distribution
6. Export exploration summary


---
## Setup


In [1]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test')
import pandas as pd
import numpy as np
from collections import Counter

from config import config
from utils import (
    DatabaseManager, 
    log_step, 
    Timer,
    save_json,
    field_coverage_table,
    describe_dataframe
)
from data_prep import (
    load_dim_org,
    load_grid,
    load_mismatched,
    create_unified_schema,
    filter_bad_records,
    is_generic_name
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)

print("Imports loaded successfully")


Imports loaded successfully


In [2]:
# Initialize database connection
db = DatabaseManager()
print("Database manager ready")


Database manager ready


---
## 1. Load Sample Data


In [3]:
# Configuration for exploration
SAMPLE_SIZE = config.sampling.EXPLORATION_SAMPLE_PER_SOURCE
print(f"Exploration sample size: {SAMPLE_SIZE} per source")


Exploration sample size: 500 per source


In [4]:
# Load dim_organization sample
dim_org_raw = load_dim_org(db, sample_size=105000)
print(f"\ndim_organization: {len(dim_org_raw):,} rows")
dim_org_raw.head(3)


[14:22:22]  Starting: Loading dim_organization
[14:22:22]  Executing: dim_organization (limit=105000)
[14:22:29]    Returned 105,000 rows in 6.5s
[14:22:29]  Completed: Loading dim_organization (6.8s)

dim_organization: 105,000 rows


,allsci_id,name,type,country_code,city,latitude,longitude,name_aliases,ror_id,grid_id,source
0,ASC-OR-0000000000004-1.0-1724880238,Living Tongues Institute for Endangered Languages,nonprofit,US,Salem,44.886715,-123.031906,"[Living Tongues Institute for Endangered Languages, LTIEL]",https://ror.org/00gens972,grid.487621.e,dim_org
1,ASC-OR-0000000000012-1.0-1724880238,International Association of Comparative Korean Studies,other,KR,Incheon,37.452202,126.653313,"[International Association of Comparative Korean Studies, IACKS, 국제비교한국학회]",https://ror.org/004427313,grid.496049.6,dim_org
2,ASC-OR-0000000000015-1.0-1724880238,Sunol Sciences Corporation (United States),company,US,Dublin,37.710030,-121.914993,[Sunol Sciences Corporation (United States)],https://ror.org/002a8dp29,grid.456183.d,dim_org


In [5]:
# Load GRID sample
grid_raw = load_grid(db, sample_size=105000)
print(f"\nGRID: {len(grid_raw):,} rows")
grid_raw.head(3)


[14:22:29]  Starting: Loading GRID data
[14:22:29]  Executing: GRID (limit=105000)
[14:22:34]    Returned 105,000 rows in 5.4s
[14:22:34]  Completed: Loading GRID data (5.4s)

GRID: 105,000 rows


,grid_id,name,aliases,acronyms,types,city,country_code,latitude,longitude,ror_id,organization_parent_ids,organization_child_ids,source
0,grid.444802.e,Imam Reza International University,[],[IRIU],[Education],Mashhad,IR,36.295559,59.592823,https://ror.org/007jfm765,[],[],grid
1,grid.490067.c,Shahid Kamyab Hospital,[],[],[Healthcare],Mashhad,IR,36.269077,59.608650,https://ror.org/02tj1w674,[],[],grid
2,grid.473805.d,Tabaran Institute of Higher Education,[],[],[Education],Mashhad,IR,36.350922,59.492165,https://ror.org/01qpg5w50,[],[],grid


In [6]:
grid_raw['ror_id'].isna().sum()

np.int64(7088)

In [7]:
# Load mismatched sample (stratified by source)
mismatched_raw = load_mismatched(db, samples_per_source=SAMPLE_SIZE)
print(f"\nMismatched: {len(mismatched_raw):,} rows")
print(f"\nSource tables:")
print(mismatched_raw['source_table'].value_counts())


[14:22:34]  Starting: Loading potential_mismatched_organizations
[14:22:34]  Executing: mismatched (500 per source)
[14:22:46]    Returned 5,000 rows in 12.1s
[14:22:46]  Parsing metadata...
[14:22:46]    Extracted 5,000 records with metadata fields
[14:22:46]  Completed: Loading potential_mismatched_organizations (12.3s)

Mismatched: 5,000 rows

Source tables:
source_table
nih_clinical_trials_gov_silver.cl_trial_organizations    500
chinese_clinical_trials_silver.trials                    500
chinese_clinical_trials_silver.trial_contacts            500
nih_clinical_trials_gov_silver.cl_trial_collaborators    500
open_fda_silver.ndc_drugs                                500
uspto_silver.patents                                     500
legacy_alpha_silver._organizations_consolidated          500
who_clinical_trials_silver.studies_metadata              500
nih_clinical_trials_gov_silver.cl_trial_locations        500
nih_clinical_trials_gov_silver.cl_trial_sponsors         500
Name: count, 

---
## 2. Field Coverage Comparison


In [8]:
# Key fields to analyze
KEY_FIELDS = [
    'name', 'name_clean', 'name_normalized', 
    'name_prefix_5', 'name_prefix_10',
    'org_type', 'country_code', 'city',
    'latitude', 'longitude',
    'ror_id', 'grid_id'
]


In [9]:
# Create unified schemas for comparison
dim_org_unified = create_unified_schema(dim_org_raw, 'dim_org')
grid_unified = create_unified_schema(grid_raw, 'grid')
mismatched_unified = create_unified_schema(mismatched_raw, 'mismatched')


[14:22:46]  Creating unified schema for dim_org...
[14:22:51]    Created unified schema: 105,000 rows, 16 columns
[14:22:51]  Creating unified schema for grid...
[14:22:55]    Created unified schema: 105,000 rows, 16 columns
[14:22:55]  Creating unified schema for mismatched...
[14:22:57]    Created unified schema: 5,000 rows, 18 columns


In [10]:
# Field coverage for each table
print("FIELD COVERAGE COMPARISON")
print("=" * 70)

coverage_dim = field_coverage_table(dim_org_unified, KEY_FIELDS)
coverage_grid = field_coverage_table(grid_unified, KEY_FIELDS)
coverage_mis = field_coverage_table(mismatched_unified, KEY_FIELDS)

# Combine into single view
coverage_combined = pd.DataFrame({
    'field': KEY_FIELDS,
    'dim_org_%': [coverage_dim[coverage_dim['field']==f]['coverage_pct'].values[0] for f in KEY_FIELDS],
    'grid_%': [coverage_grid[coverage_grid['field']==f]['coverage_pct'].values[0] for f in KEY_FIELDS],
    'mismatched_%': [coverage_mis[coverage_mis['field']==f]['coverage_pct'].values[0] for f in KEY_FIELDS]
})

coverage_combined


FIELD COVERAGE COMPARISON


,field,dim_org_%,grid_%,mismatched_%
0,name,100.0,100.0,93.9
1,name_clean,100.0,100.0,93.9
2,name_normalized,100.0,100.0,93.9
3,name_prefix_5,100.0,100.0,93.9
4,name_prefix_10,100.0,100.0,93.9
5,org_type,99.5,100.0,34.4
6,country_code,99.8,100.0,19.5
7,city,99.5,100.0,9.9
8,latitude,99.0,99.4,9.7
9,longitude,99.0,99.4,9.7


In [11]:
# Visual coverage comparison
print("\nVISUAL FIELD COVERAGE")
print("=" * 70)
print(f"{'Field':<20} | {'dim_org':<15} | {'grid':<15} | {'mismatched':<15}")
print("-" * 70)

for _, row in coverage_combined.iterrows():
    def bar(pct):
        filled = int(pct / 10)
        return '#' * filled + '.' * (10 - filled) + f" {pct:5.1f}%"
    
    print(f"{row['field']:<20} | {bar(row['dim_org_%']):<15} | {bar(row['grid_%']):<15} | {bar(row['mismatched_%']):<15}")



VISUAL FIELD COVERAGE
Field                | dim_org         | grid            | mismatched     
----------------------------------------------------------------------
name                 | ########## 100.0% | ########## 100.0% | #########.  93.9%
name_clean           | ########## 100.0% | ########## 100.0% | #########.  93.9%
name_normalized      | ########## 100.0% | ########## 100.0% | #########.  93.9%
name_prefix_5        | ########## 100.0% | ########## 100.0% | #########.  93.9%
name_prefix_10       | ########## 100.0% | ########## 100.0% | #########.  93.9%
org_type             | #########.  99.5% | ########## 100.0% | ###.......  34.4%
country_code         | #########.  99.8% | ########## 100.0% | #.........  19.5%
city                 | #########.  99.5% | ########## 100.0% | ..........   9.9%
latitude             | #########.  99.0% | #########.  99.4% | ..........   9.7%
longitude            | #########.  99.0% | #########.  99.4% | ..........   9.7%
ror_id               

---
## 3. ROR ID Overlap Analysis (Ground Truth)


In [12]:
# Count records with ROR IDs
dim_with_ror = dim_org_unified[dim_org_unified['ror_id'].notna()]
grid_with_ror = grid_unified[grid_unified['ror_id'].notna()]

print("ROR ID AVAILABILITY")
print("=" * 50)
print(f"dim_organization with ROR: {len(dim_with_ror):,} / {len(dim_org_unified):,} ({100*len(dim_with_ror)/len(dim_org_unified):.1f}%)")
print(f"GRID with ROR:             {len(grid_with_ror):,} / {len(grid_unified):,} ({100*len(grid_with_ror)/len(grid_unified):.1f}%)")


ROR ID AVAILABILITY
dim_organization with ROR: 102,943 / 105,000 (98.0%)
GRID with ROR:             97,912 / 105,000 (93.2%)


In [13]:
# Find overlapping ROR IDs (these become ground truth pairs)
dim_ror_set = set(dim_with_ror['ror_id'].dropna())
grid_ror_set = set(grid_with_ror['ror_id'].dropna())

overlap_ror = dim_ror_set & grid_ror_set

print("\nROR ID OVERLAP (Ground Truth Potential)")
print("=" * 50)
print(f"Unique ROR IDs in dim_org: {len(dim_ror_set):,}")
print(f"Unique ROR IDs in GRID:    {len(grid_ror_set):,}")
print(f"Overlapping ROR IDs:       {len(overlap_ror):,}")
print(f"")
print(f"This gives us {len(overlap_ror):,} ground truth matches for training!")



ROR ID OVERLAP (Ground Truth Potential)
Unique ROR IDs in dim_org: 102,943
Unique ROR IDs in GRID:    97,912
Overlapping ROR IDs:       91,939

This gives us 91,939 ground truth matches for training!


In [14]:
# Sample of matching pairs
if len(overlap_ror) > 0:
    print("\nSAMPLE GROUND TRUTH PAIRS")
    print("=" * 80)
    
    sample_rors = list(overlap_ror)[:5]
    for ror in sample_rors:
        dim_name = dim_with_ror[dim_with_ror['ror_id'] == ror]['name'].iloc[0]
        grid_name = grid_with_ror[grid_with_ror['ror_id'] == ror]['name'].iloc[0]
        print(f"\nROR: {ror}")
        print(f"  dim_org: {dim_name[:60]}")
        print(f"  GRID:    {grid_name[:60]}")



SAMPLE GROUND TRUTH PAIRS

ROR: https://ror.org/02edr8z79
  dim_org: Thailand Graduate Institute of Science and Technology
  GRID:    Thailand Graduate Institute of Science and Technology

ROR: https://ror.org/01f5v9z11
  dim_org: Swedish Network for Innovation & Technology Transfer Support
  GRID:    Swedish Network for Innovation & Technology Transfer Support

ROR: https://ror.org/03y5bcg65
  dim_org: Action for Trans Health
  GRID:    Action for Trans Health

ROR: https://ror.org/03r72d818
  dim_org: Hokuto (Japan)
  GRID:    Hokuto (Japan)

ROR: https://ror.org/01da73p86
  dim_org: Keep-it Technologies (Norway)
  GRID:    Keep-it Technologies (Norway)


---
## 4. Blocking Rule Coverage Simulation


In [15]:
# Analyze blocking key distributions
print("BLOCKING KEY ANALYSIS")
print("=" * 60)

def analyze_blocking_key(df, key_col, name):
    """Analyze a blocking key column"""
    non_null = df[key_col].notna().sum()
    coverage = 100 * non_null / len(df)
    unique_values = df[key_col].nunique()
    
    # Estimate pairs generated
    value_counts = df[key_col].value_counts()
    pairs_per_key = (value_counts * (value_counts - 1) / 2).sum()
    
    print(f"\n{name}:")
    print(f"  Coverage: {non_null:,} / {len(df):,} ({coverage:.1f}%)")
    print(f"  Unique values: {unique_values:,}")
    print(f"  Estimated pairs: {pairs_per_key:,.0f}")
    print(f"  Top values: {value_counts.head(5).to_dict()}")
    
    return {
        'key': name,
        'coverage_pct': coverage,
        'unique_values': unique_values,
        'estimated_pairs': pairs_per_key
    }

blocking_stats = []
for df, source in [(dim_org_unified, 'dim_org'), (grid_unified, 'grid'), (mismatched_unified, 'mismatched')]:
    print(f"\n{'='*60}")
    print(f"SOURCE: {source}")
    print(f"{'='*60}")
    
    for key in ['name_prefix_5', 'name_prefix_10', 'country_code']:
        if key in df.columns:
            stats = analyze_blocking_key(df, key, key)
            stats['source'] = source
            blocking_stats.append(stats)


BLOCKING KEY ANALYSIS

SOURCE: dim_org

name_prefix_5:
  Coverage: 104,994 / 105,000 (100.0%)
  Unique values: 29,477
  Estimated pairs: 20,824,242
  Top values: {'unive': 3675, 'insti': 3634, 'natio': 1942, 'centr': 1938, 'inter': 977}

name_prefix_10:
  Coverage: 104,994 / 105,000 (100.0%)
  Unique values: 60,408
  Estimated pairs: 5,756,897
  Top values: {'university': 1810, 'institute ': 1731, 'universida': 1062, 'instituto ': 833, 'internatio': 781}

country_code:
  Coverage: 104,828 / 105,000 (99.8%)
  Unique values: 229
  Estimated pairs: 561,390,369
  Top values: {'US': 30274, 'GB': 7263, 'DE': 5052, 'CN': 4685, 'FR': 4628}

SOURCE: grid

name_prefix_5:
  Coverage: 104,994 / 105,000 (100.0%)
  Unique values: 31,076
  Estimated pairs: 18,184,159
  Top values: {'insti': 3459, 'unive': 3343, 'natio': 1928, 'centr': 1633, 'inter': 965}

name_prefix_10:
  Coverage: 104,994 / 105,000 (100.0%)
  Unique values: 62,545
  Estimated pairs: 5,427,857
  Top values: {'institute ': 1827, 'uni

In [16]:
# Estimate cross-table blocking coverage
print("\nCROSS-TABLE BLOCKING SIMULATION")
print("=" * 60)

# How many mismatched records would be blocked with training data?
for key in ['name_prefix_5', 'name_prefix_10', 'country_code']:
    if key in mismatched_unified.columns and key in dim_org_unified.columns:
        mis_keys = set(mismatched_unified[key].dropna())
        dim_keys = set(dim_org_unified[key].dropna())
        
        overlap = len(mis_keys & dim_keys)
        mis_coverage = 100 * overlap / len(mis_keys) if mis_keys else 0
        
        print(f"\n{key}:")
        print(f"  Mismatched keys: {len(mis_keys):,}")
        print(f"  dim_org keys:    {len(dim_keys):,}")
        print(f"  Overlap:         {overlap:,} ({mis_coverage:.1f}% of mismatched)")



CROSS-TABLE BLOCKING SIMULATION

name_prefix_5:
  Mismatched keys: 1,943
  dim_org keys:    29,477
  Overlap:         1,195 (61.5% of mismatched)

name_prefix_10:
  Mismatched keys: 2,679
  dim_org keys:    60,408
  Overlap:         807 (30.1% of mismatched)

country_code:
  Mismatched keys: 63
  dim_org keys:    229
  Overlap:         63 (100.0% of mismatched)


In [17]:
# Identify records with NO blocking key
no_blocking_key = mismatched_unified[
    mismatched_unified['name_prefix_5'].isna() & 
    mismatched_unified['name_prefix_10'].isna() &
    mismatched_unified['country_code'].isna()
]

print(f"\nRECORDS WITH NO USABLE BLOCKING KEY")
print(f"=" * 50)
print(f"Count: {len(no_blocking_key):,} / {len(mismatched_unified):,} ({100*len(no_blocking_key)/len(mismatched_unified):.1f}%)")

if len(no_blocking_key) > 0:
    print(f"\nSamples:")
    display(no_blocking_key[['unique_id', 'name', 'source']].head(10))



RECORDS WITH NO USABLE BLOCKING KEY
Count: 282 / 5,000 (5.6%)

Samples:


,unique_id,name,source
1500,mis_nan,NaN,mismatched
1501,mis_nan,NaN,mismatched
1503,mis_nan,NaN,mismatched
1506,mis_nan,NaN,mismatched
1507,mis_nan,NaN,mismatched
1509,mis_nan,NaN,mismatched
1514,mis_nan,NaN,mismatched
1518,mis_nan,NaN,mismatched
1520,mis_nan,NaN,mismatched
1521,mis_nan,NaN,mismatched


---
## 5. Name Quality Distribution


In [18]:
# Name length distribution
print("NAME LENGTH DISTRIBUTION")
print("=" * 50)

for df, source in [(dim_org_unified, 'dim_org'), (grid_unified, 'grid'), (mismatched_unified, 'mismatched')]:
    lengths = df['name'].str.len()
    print(f"\n{source}:")
    print(f"  Min: {lengths.min()}, Max: {lengths.max()}")
    print(f"  Mean: {lengths.mean():.1f}, Median: {lengths.median():.1f}")
    print(f"  Very short (<5): {(lengths < 5).sum():,}")
    print(f"  Very long (>100): {(lengths > 100).sum():,}")


NAME LENGTH DISTRIBUTION

dim_org:
  Min: 2, Max: 191
  Mean: 31.1, Median: 29.0
  Very short (<5): 133
  Very long (>100): 116

grid:
  Min: 2, Max: 191
  Mean: 30.3, Median: 28.0
  Very short (<5): 139
  Very long (>100): 88

mismatched:
  Min: 3.0, Max: 181.0
  Mean: 37.1, Median: 33.0
  Very short (<5): 5
  Very long (>100): 59


In [19]:
# Generic name detection in mismatched
print("\nGENERIC NAME DETECTION (mismatched only)")
print("=" * 50)

mismatched_unified['is_generic'] = mismatched_unified['name'].apply(is_generic_name)
generic_count = mismatched_unified['is_generic'].sum()
print(f"Generic names: {generic_count:,} / {len(mismatched_unified):,} ({100*generic_count/len(mismatched_unified):.1f}%)")

if generic_count > 0:
    print(f"\nExamples:")
    display(mismatched_unified[mismatched_unified['is_generic']][['name', 'source']].head(10))



GENERIC NAME DETECTION (mismatched only)
Generic names: 56 / 5,000 (1.1%)

Examples:


,name,source
4000,Research Site,mismatched
4010,Novartis Investigative Site,mismatched
4016,Research Site,mismatched
4026,Novartis Investigative Site,mismatched
4044,Research Site,mismatched
4049,Novartis Investigative Site,mismatched
4066,Research Site,mismatched
4078,Research Site,mismatched
4082,Novartis Investigative Site,mismatched
4097,Research Site,mismatched


In [20]:
# Most common names (potential duplicates or term frequency issues)
print("\nMOST COMMON NAMES")
print("=" * 50)

for df, source in [(dim_org_unified, 'dim_org'), (mismatched_unified, 'mismatched')]:
    name_counts = df['name'].value_counts().head(10)
    print(f"\n{source}:")
    for name, count in name_counts.items():
        if count > 1:
            print(f"  {count:>4}x | {name[:60]}")



MOST COMMON NAMES

dim_org:
    51x | Ministry of Health
    21x | Ministry of Education
    16x | Ministry of Justice
    13x | Ministry of Foreign Affairs
    13x | Government Medical College
    12x | St. Luke's Hospital
    11x | Ministry of Culture
    10x | St Mary's Hospital
    10x | Ministry of Agriculture
    10x | Ministry of Finance

mismatched:
    35x | Research Site
    29x | National Cancer Institute (NCI)
    27x | National Institutes of Health Clinical Center (CC)
    19x | Hahnemann Laboratories, INC.
    16x | Novartis Investigative Site
    16x | M.D. Anderson Cancer Center
    14x | A-S Medication Solutions
    13x | Bryant Ranch Prepack
    13x | Merck Sharp & Dohme LLC
    12x | GlaxoSmithKline


In [21]:
# Organization category analysis (sub-orgs vs standard orgs)
print("\nORGANIZATION CATEGORIES")
print("=" * 50)

def classify_org(name):
    if pd.isna(name):
        return 'Unknown'
    name_lower = name.lower()
    if 'department' in name_lower:
        return 'Department'
    elif 'division' in name_lower:
        return 'Division'
    elif 'center' in name_lower or 'centre' in name_lower:
        return 'Center'
    elif 'institute' in name_lower:
        return 'Institute'
    elif 'school of' in name_lower:
        return 'School'
    elif 'university' in name_lower:
        return 'University'
    elif 'hospital' in name_lower:
        return 'Hospital'
    else:
        return 'Other'

mismatched_unified['org_category'] = mismatched_unified['name'].apply(classify_org)
category_counts = mismatched_unified['org_category'].value_counts()

for cat, count in category_counts.items():
    pct = 100 * count / len(mismatched_unified)
    print(f"  {cat:<15} | {count:>6,} ({pct:5.1f}%)")



ORGANIZATION CATEGORIES
  Other           |  2,535 ( 50.7%)
  University      |    866 ( 17.3%)
  Hospital        |    595 ( 11.9%)
  Center          |    325 (  6.5%)
  Unknown         |    305 (  6.1%)
  Institute       |    161 (  3.2%)
  School          |    105 (  2.1%)
  Department      |    105 (  2.1%)
  Division        |      3 (  0.1%)


---
## 6. Filter Impact Analysis


In [22]:
# Test filtering on mismatched data
print("FILTER IMPACT ANALYSIS")
print("=" * 60)

filtered_df, removed_df = filter_bad_records(mismatched_unified)


FILTER IMPACT ANALYSIS
[14:22:57]  Filtering bad records...
[14:22:57]    Removed 366 of 5,000 records (7.3%)
[14:22:57]      - null_name: 305
[14:22:57]      - generic_name: 56
[14:22:57]      - short_name: 5


In [23]:
# Removal breakdown
if '_remove_reason' in removed_df.columns:
    print("\nRemoval reasons:")
    print(removed_df['_remove_reason'].value_counts())

# Sample removed records
if len(removed_df) > 0:
    print("\nSample removed records:")
    display(removed_df[['unique_id', 'name', '_remove_reason']].head(10))



Removal reasons:
_remove_reason
null_name       305
generic_name     56
short_name        5
Name: count, dtype: int64

Sample removed records:


,unique_id,name,_remove_reason
83,mis_aacbf166bbb9e73c8579e7dd3e5ddd38,RAND,short_name
1500,mis_nan,NaN,null_name
1501,mis_nan,NaN,null_name
1503,mis_nan,NaN,null_name
1506,mis_nan,NaN,null_name
1507,mis_nan,NaN,null_name
1509,mis_nan,NaN,null_name
1514,mis_nan,NaN,null_name
1516,mis_ef45df20f2fd57b67a8fa05b6e3eea1b,Rumb,short_name
1518,mis_nan,NaN,null_name


---
## 7. Export Exploration Summary


In [24]:
# Compile summary statistics
exploration_summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    
    'data_sizes': {
        'dim_org_sample': len(dim_org_unified),
        'grid_sample': len(grid_unified),
        'mismatched_sample': len(mismatched_unified)
    },
    
    'ror_overlap': {
        'dim_org_with_ror': len(dim_with_ror),
        'grid_with_ror': len(grid_with_ror),
        'overlapping_ror_ids': len(overlap_ror),
        'ground_truth_pairs_available': len(overlap_ror)
    },
    
    'field_coverage': coverage_combined.to_dict('records'),
    
    'blocking_stats': blocking_stats,
    
    'name_quality': {
        'generic_names_count': int(generic_count),
        'generic_names_pct': float(100 * generic_count / len(mismatched_unified)),
        'no_blocking_key_count': len(no_blocking_key),
        'no_blocking_key_pct': float(100 * len(no_blocking_key) / len(mismatched_unified))
    },
    
    'filtering': {
        'records_after_filter': len(filtered_df),
        'records_removed': len(removed_df),
        'removal_pct': float(100 * len(removed_df) / len(mismatched_unified))
    }
}

save_json(exploration_summary, config.paths.EXPLORATION_SUMMARY, "Exploration summary")


[14:22:57]  Saving JSON: Exploration summary
[14:22:57]    Saved to /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/exploration_summary.json


In [25]:
# Print key findings
print("\n" + "=" * 70)
print("KEY FINDINGS SUMMARY")
print("=" * 70)

print(f"""
DATA SIZES:
  - dim_organization sample: {len(dim_org_unified):,} records
  - GRID sample: {len(grid_unified):,} records
  - Mismatched sample: {len(mismatched_unified):,} records

GROUND TRUTH (ROR Overlap):
  - dim_org with ROR: {100*len(dim_with_ror)/len(dim_org_unified):.1f}%
  - GRID with ROR: {100*len(grid_with_ror)/len(grid_unified):.1f}%
  - Overlapping ROR IDs: {len(overlap_ror):,} (available for training)

BLOCKING COVERAGE:
  - Records with no blocking key: {len(no_blocking_key):,} ({100*len(no_blocking_key)/len(mismatched_unified):.1f}%)

DATA QUALITY:
  - Generic names: {generic_count:,} ({100*generic_count/len(mismatched_unified):.1f}%)
  - Records removed by filters: {len(removed_df):,} ({100*len(removed_df)/len(mismatched_unified):.1f}%)
  - Records remaining for matching: {len(filtered_df):,}

RECOMMENDATIONS:
  1. Use ROR matches as ground truth for training
  2. Filter out generic names before matching
  3. Use multiple blocking rules to maximize coverage
  4. Consider feature-sparse training for inference robustness
""")



KEY FINDINGS SUMMARY

DATA SIZES:
  - dim_organization sample: 105,000 records
  - GRID sample: 105,000 records
  - Mismatched sample: 5,000 records

GROUND TRUTH (ROR Overlap):
  - dim_org with ROR: 98.0%
  - GRID with ROR: 93.2%
  - Overlapping ROR IDs: 91,939 (available for training)

BLOCKING COVERAGE:
  - Records with no blocking key: 282 (5.6%)

DATA QUALITY:
  - Generic names: 56 (1.1%)
  - Records removed by filters: 366 (7.3%)
  - Records remaining for matching: 4,634

RECOMMENDATIONS:
  1. Use ROR matches as ground truth for training
  2. Filter out generic names before matching
  3. Use multiple blocking rules to maximize coverage
  4. Consider feature-sparse training for inference robustness



In [26]:
# Cleanup
db.close()
print("\nExploration complete. Summary saved to:", config.paths.EXPLORATION_SUMMARY)



Exploration complete. Summary saved to: /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/exploration_summary.json
